# 🛍️ Customer Analysis Project
**Tools:** Python, pandas, scikit-learn, seaborn
**Dataset:** 5,630 E-commerce Customers
**Goal:** Understand customer behaviour, segment customers, and predict churn

In [ ]:
# CELL 1 — Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries loaded!')

## Phase 1 — Data Loading & Exploration

In [ ]:
# CELL 2 — Load dataset
import kagglehub, os
path = kagglehub.dataset_download('ankitverma2010/ecommerce-customer-churn-analysis-and-prediction')
df = pd.read_excel(path + '/E Commerce Dataset.xlsx', sheet_name='E Comm')
print(f'Dataset loaded! Shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

In [ ]:
# CELL 3 — Data types and info
df.info()

In [ ]:
# CELL 4 — Summary statistics
df.describe().round(2)

In [ ]:
# CELL 5 — Check missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct}).sort_values('Missing %', ascending=False)
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# CELL 6 — Churn distribution
churn_counts = df['Churn'].value_counts()
print(f'Active customers:  {churn_counts[0]} ({churn_counts[0]/len(df)*100:.1f}%)')
print(f'Churned customers: {churn_counts[1]} ({churn_counts[1]/len(df)*100:.1f}%)')

plt.figure(figsize=(6,4))
sns.countplot(x='Churn', data=df, palette=['#4CAF50','#F44336'])
plt.title('Customer Churn Distribution')
plt.xticks([0,1], ['Active','Churned'])
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.savefig('churn_distribution.png', dpi=150)
plt.show()

## Phase 2 — Data Cleaning

In [ ]:
# CELL 7 — Fill missing values with median
cols_to_fill = ['Tenure','WarehouseToHome','HourSpendOnApp',
                'OrderAmountHikeFromlastYear','CouponUsed','OrderCount','DaySinceLastOrder']

for col in cols_to_fill:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"Filled '{col}' with median: {median_val}")

print(f'\nTotal missing values remaining: {df.isnull().sum().sum()}')

In [ ]:
# CELL 8 — Save clean data
df.to_csv('customer_clean.csv', index=False)
print('Clean data saved!')

## Phase 3 — RFM Customer Segmentation

In [ ]:
# CELL 9 — Build RFM table
rfm = df[['CustomerID','DaySinceLastOrder','OrderCount','CashbackAmount']].copy()
rfm.columns = ['CustomerID','Recency','Frequency','Monetary']

rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5,4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), q=5, labels=[1,2,3,4,5])

def segment_customer(row):
    score = int(row['R_Score']) + int(row['F_Score']) + int(row['M_Score'])
    if score >= 13: return 'Champion'
    elif score >= 10: return 'Loyal'
    elif score >= 7: return 'At Risk'
    else: return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
print(rfm['Segment'].value_counts())
rfm.head()

In [ ]:
# CELL 10 — Plot RFM segments
colors = ['#3498db','#2ecc71','#e67e22','#e74c3c']
rfm['Segment'].value_counts().plot(kind='bar', color=colors)
plt.title('Customer Segments (RFM Analysis)')
plt.xlabel('Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('rfm_segments.png', dpi=150)
plt.show()

## Phase 4 — Churn Prediction Model

In [ ]:
# CELL 11 — Encode and train model
df_model = df.copy()
text_cols = ['PreferredLoginDevice','PreferredPaymentMode','Gender','PreferedOrderCat','MaritalStatus']
le = LabelEncoder()
for col in text_cols:
    df_model[col] = le.fit_transform(df_model[col])

X = df_model.drop(['CustomerID','Churn'], axis=1)
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
# CELL 12 — Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Active','Churned'], yticklabels=['Active','Churned'])
plt.title('Churn Prediction — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## Phase 5 — Dashboard

In [ ]:
# CELL 13 — Python dashboard (4 charts)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Customer Analysis Dashboard', fontsize=16, fontweight='bold')

# Chart 1 — Churn
churn_counts = df['Churn'].value_counts()
axes[0,0].pie(churn_counts, labels=['Active','Churned'], autopct='%1.1f%%', colors=['#2ecc71','#e74c3c'])
axes[0,0].set_title('Churn Distribution')

# Chart 2 — RFM Segments
rfm['Segment'].value_counts().plot(kind='bar', ax=axes[0,1], color=['#3498db','#2ecc71','#e67e22','#e74c3c'])
axes[0,1].set_title('Customer Segments')
axes[0,1].tick_params(axis='x', rotation=0)

# Chart 3 — Churn by Tenure
df.groupby('Tenure')['Churn'].mean().plot(ax=axes[1,0], color='#9b59b6')
axes[1,0].set_title('Churn Rate by Tenure')
axes[1,0].set_ylabel('Churn Rate')

# Chart 4 — Cashback by Segment
rfm.groupby('Segment')['Monetary'].mean().sort_values().plot(kind='barh', ax=axes[1,1], color='#1abc9c')
axes[1,1].set_title('Avg Cashback by Segment')

plt.tight_layout()
plt.savefig('customer_dashboard.png', dpi=150)
plt.show()
print('Dashboard saved!')

## Phase 6 — Key Insights & Business Recommendations

In [ ]:
# CELL 14 — Print insights
print("""
KEY FINDINGS
============
1. CHURN RATE: 16.8% (948/5630) — below industry average of 20-30%
2. TENURE: Churn is highest in months 0-2 — onboarding is critical
3. SEGMENTS: 43% of customers are At Risk — urgent action needed
4. MODEL: 89% accuracy with Logistic Regression

RECOMMENDATIONS
===============
1. Launch onboarding email series for new customers (month 0-2)
2. Create loyalty rewards for At Risk segment
3. Run win-back campaign for 444 Lost customers
4. Deploy churn model weekly to flag at-risk customers
""")